# NIST TN 1822 — Verif.2.1: Speed in a corridor

Single agent walks 40 m at 1.0 m/s. The corridor is extended to 60 m with 10 m acceleration and 10 m isolation buffers; see MODIFICATIONS.md.

In [ ]:
from datetime import datetime
print(f"Executed on {datetime.now().astimezone().strftime('%d %B %Y, %H:%M %Z')}")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pedpy
from shapely.geometry import Point, Polygon

from jupedsim_scenarios import load_scenario, run_scenario

In [ ]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f7f7f5",
    "axes.edgecolor": "#3a3a3a",
    "axes.labelcolor": "#1d1d1d",
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.figsize": (8, 5),
})

## Load and run the scenario

In [ ]:
SCENARIO_ZIP = Path("scenario_files") / "Nist-2-1-corridor-speed.zip"
scenario = load_scenario(str(SCENARIO_ZIP))
print(scenario.summary())
result = run_scenario(scenario, seed=42)

## Measure walking speed across the 40 m segment

In [ ]:
df = result.trajectory_dataframe().sort_values(['id', 'frame'])
MEAS_START_X = 10.0
MEAS_END_X = 50.0
DIRECTION = 'right'
rows = []
for agent_id, sub in df.groupby('id'):
    sub = sub.reset_index(drop=True)
    if DIRECTION == 'right':
        t_in_idx = sub[sub.x >= MEAS_START_X].index.min()
        t_out_idx = sub[sub.x >= MEAS_END_X].index.min()
    else:
        t_in_idx = sub[sub.x <= MEAS_START_X].index.min()
        t_out_idx = sub[sub.x <= MEAS_END_X].index.min()
    if pd.isna(t_in_idx) or pd.isna(t_out_idx):
        continue
    t_in = sub.loc[t_in_idx, 'frame'] / result.frame_rate
    t_out = sub.loc[t_out_idx, 'frame'] / result.frame_rate
    rows.append({'id': int(agent_id), 't_in_s': t_in, 't_out_s': t_out, 'transit_s': t_out - t_in})
transit = pd.DataFrame(rows)
transit['speed_m_s'] = abs(MEAS_END_X - MEAS_START_X) / transit['transit_s']
transit

## Plot trajectory x vs t

In [ ]:
fig, ax = plt.subplots()
for agent_id, sub in df.groupby('id'):
    ax.plot(sub.frame / result.frame_rate, sub.x, label=f'agent {agent_id}')
ax.axhline(MEAS_START_X, color='k', ls='--', alpha=0.3, label='measure start')
ax.axhline(MEAS_END_X, color='k', ls=':', alpha=0.3, label='measure end')
ax.set_xlabel('time [s]')
ax.set_ylabel('x position [m]')
ax.legend()
plt.show()

## Acceptance

In [ ]:
TARGET = 1.0
TOL = 0.05
observed = transit['speed_m_s'].mean()
print(f'observed mean speed = {observed:.3f} m/s; target = {TARGET} m/s')
assert abs(observed - TARGET) <= TOL, (observed, TARGET)

In [ ]:
result.cleanup()